In [1]:
# ── Cell 1: Install captum + numpy (no gradio install needed) ─────────────────
import subprocess
subprocess.run(["pip", "install", "-q", "captum", "numpy==1.26.4"], check=True)
print("✅ Done. RESTART RUNTIME now, then run from Cell 2.")

✅ Done. RESTART RUNTIME now, then run from Cell 2.


In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# ── Cell 2: Imports (run after restart) ───────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import torch
import gradio as gr
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import LayerIntegratedGradients

print(f"Gradio: {gr.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


Mounted at /content/drive
Gradio: 5.50.0
GPU: Tesla T4


In [2]:
# ── Cell 3: Config & Paths ────────────────────────────────────────────────────
DATA_DIR       = "/content/drive/MyDrive/disaster_project/disaster_data"
MODELS_DIR     = "/content/drive/MyDrive/disaster_project/disaster_models"
CONFIDENCE_DIR = "/content/drive/MyDrive/disaster_project/confidence_results"
MODEL_PATH     = f"{MODELS_DIR}/roberta/best_model"
NUM_TWEETS     = 300

with open(f"{DATA_DIR}/label_mapping.json") as f:
    mapping = json.load(f)

label2id    = mapping["label2id"]
id2label    = {int(k): v for k, v in mapping["id2label"].items()}
NUM_LABELS  = len(label2id)
CLASS_NAMES = [id2label[i] for i in range(NUM_LABELS)]

OPTIMAL_THRESHOLD = float(np.load(f"{CONFIDENCE_DIR}/optimal_threshold.npy"))

CLASS_ICONS = {
    "caution_and_advice":                     "⚠️",
    "displaced_people_and_evacuations":        "🚶",
    "infrastructure_and_utility_damage":       "🏚️",
    "injured_or_dead_people":                  "🏥",
    "missing_or_found_people":                 "🔍",
    "not_humanitarian":                        "🚫",
    "other_relevant_information":              "ℹ️",
    "requests_or_urgent_needs":                "🆘",
    "rescue_volunteering_or_donation_effort":  "🤝",
    "sympathy_and_support":                    "💙",
}

CLASS_COLORS = {
    "caution_and_advice":                     "#f0a500",
    "displaced_people_and_evacuations":        "#7c6fe0",
    "infrastructure_and_utility_damage":       "#e07c3a",
    "injured_or_dead_people":                  "#e05555",
    "missing_or_found_people":                 "#e0c155",
    "not_humanitarian":                        "#6b7a8a",
    "other_relevant_information":              "#5b8dd9",
    "requests_or_urgent_needs":                "#e03a7c",
    "rescue_volunteering_or_donation_effort":  "#00c47c",
    "sympathy_and_support":                    "#5bc4e0",
}

FONTS = '<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;600;700&display=swap" rel="stylesheet">'

print(f"Config loaded. Threshold: {OPTIMAL_THRESHOLD}")

Config loaded. Threshold: 0.74


In [3]:
# ── Cell 4: Load Model ────────────────────────────────────────────────────────
print("\nLoading RoBERTa...")
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, num_labels=NUM_LABELS,
    id2label=id2label, label2id=label2id,
)
model.to(device)
model.eval()

lig = LayerIntegratedGradients(
    lambda input_ids, attn_mask: model(
        input_ids=input_ids, attention_mask=attn_mask).logits,
    model.roberta.embeddings.word_embeddings,
)
print("✅ RoBERTa loaded.")


Loading RoBERTa...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ RoBERTa loaded.


In [4]:
# ── Cell 5: Inference & Attribution ──────────────────────────────────────────
def classify_tweet(text):
    enc = tokenizer(text, return_tensors="pt",
                    truncation=True, max_length=128, padding="max_length")
    ids  = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    with torch.no_grad():
        probs = torch.softmax(
            model(input_ids=ids, attention_mask=mask).logits, dim=-1
        ).squeeze(0).cpu().numpy()
    pred = int(np.argmax(probs))
    return probs, pred, float(probs[pred])


def get_attributions(text, pred_class, n_steps=50):
    enc  = tokenizer(text, return_tensors="pt",
                     truncation=True, max_length=128, padding="max_length")
    ids  = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    base = torch.zeros_like(ids)
    base[:, 0]  = tokenizer.cls_token_id
    base[:, -1] = tokenizer.sep_token_id
    attrs, _ = lig.attribute(
        inputs=ids, baselines=base,
        additional_forward_args=(mask,),
        target=pred_class, n_steps=n_steps,
        return_convergence_delta=True,
    )
    scores  = attrs.sum(dim=-1).squeeze(0).detach().cpu().numpy()
    seq_len = mask.squeeze(0).sum().item()
    tokens  = tokenizer.convert_ids_to_tokens(ids.squeeze(0).cpu().numpy())[:seq_len]
    scores  = scores[:seq_len]
    mx = np.abs(scores).max()
    if mx > 0:
        scores /= mx
    return tokens, scores

In [5]:
# ── Cell 6: Pre-classify Tweet Pool ──────────────────────────────────────────
print("\nLoading tweets...")
raw = load_dataset("parquet",
                   data_files={"test": f"{DATA_DIR}/test.parquet"})["test"]
df  = raw.to_pandas()
df  = df[df["low_info"] == False].reset_index(drop=True)

pool_dfs = []
per_class = NUM_TWEETS // NUM_LABELS
for c in range(NUM_LABELS):
    sub = df[df["label"] == c]
    pool_dfs.append(sub.sample(n=min(per_class, len(sub)), random_state=42))

pool_df = pd.concat(pool_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Pool: {len(pool_df)} tweets. Pre-classifying...")

classified_tweets = []
for i, row in pool_df.iterrows():
    probs, pred, conf = classify_tweet(row["tweet_text"])
    classified_tweets.append({
        "id":         i,
        "text":       row["tweet_text"],
        "true_label": int(row["label"]),
        "pred_label": pred,
        "pred_name":  CLASS_NAMES[pred],
        "confidence": conf,
        "uncertain":  conf < OPTIMAL_THRESHOLD,
        "probs":      probs.tolist(),
    })
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(pool_df)}...")

print(f"✅ {len(classified_tweets)} tweets ready.")


Loading tweets...


Generating test split: 0 examples [00:00, ? examples/s]

Pool: 300 tweets. Pre-classifying...
  50/300...
  100/300...
  150/300...
  200/300...
  250/300...
  300/300...
✅ 300 tweets ready.


In [6]:
# ── Cell 7: App State (simple dict, thread-safe enough for demo) ──────────────
app_state = {
    "visible_count": 0,
    "running":       False,
    "active_filter": "ALL",
    "selected_id":   None,
}

In [7]:
# ── Cell 8: HTML Builders ─────────────────────────────────────────────────────
def conf_color(c):
    if c >= OPTIMAL_THRESHOLD + 0.10: return "#00e5a0"
    if c >= OPTIMAL_THRESHOLD:        return "#f0a500"
    return "#ff5555"

def build_metrics():
    vis   = classified_tweets[:app_state["visible_count"]]
    total = len(vis)
    conf  = sum(1 for t in vis if not t["uncertain"])
    unc   = total - conf
    cpct  = f"{conf/total*100:.0f}%" if total else "—"
    top   = Counter(t["pred_name"] for t in vis if not t["uncertain"]).most_common(1)
    top_l = top[0][0].replace("_", " ") if top else "—"
    top_i = CLASS_ICONS.get(top[0][0], "") if top else ""
    top_c = CLASS_COLORS.get(top[0][0], "#e6edf3") if top else "#e6edf3"

    def box(val, lbl, color="#e6edf3"):
        return f"""<div style="background:#161b22;border:1px solid #21262d;
            border-radius:8px;padding:12px 16px;flex:1;min-width:110px;">
            <div style="font-family:'Space Mono',monospace;font-size:20px;
                font-weight:700;color:{color};">{val}</div>
            <div style="font-size:9px;color:#4a5a6a;font-family:'Space Mono',monospace;
                letter-spacing:1px;text-transform:uppercase;margin-top:3px;">{lbl}</div>
        </div>"""

    return f"""{FONTS}<div style="display:flex;gap:8px;flex-wrap:wrap;padding:4px 0;">
        {box(total, "Tweets Processed")}
        {box(cpct,  "Confident", "#00e5a0")}
        {box(unc,   "Needs Review", "#ff5555" if unc > 0 else "#4a5a6a")}
        {box(f"{top_i} {top_l}", "Most Reported", top_c)}
        {box(f"{OPTIMAL_THRESHOLD:.0%}", "Threshold", "#f0a500")}
    </div>"""

def build_sidebar():
    vis    = classified_tweets[:app_state["visible_count"]]
    total  = len(vis)
    counts = Counter(t["pred_name"] for t in vis)
    filt   = app_state["active_filter"]
    unc    = sum(1 for t in vis if t["uncertain"])

    def row(val, label, icon, color, key, extra=""):
        active = "border-color:" + color + ";" if filt == key else ""
        return f"""<div style="padding:8px 10px;border-radius:7px;cursor:pointer;
            border:1px solid #21262d;margin-bottom:4px;{active}
            display:flex;align-items:center;gap:8px;" id="sb_{key}">{extra}
            <span style="font-size:13px;">{icon}</span>
            <span style="font-family:'DM Sans',sans-serif;font-size:12px;
                color:#c9d1d9;flex:1;white-space:nowrap;overflow:hidden;
                text-overflow:ellipsis;">{label}</span>
            <span style="font-family:'Space Mono',monospace;font-size:11px;
                color:{color};font-weight:700;">{val}</span>
        </div>"""

    rows = row(total, "All Tweets",    "🌐", "#00e5a0",  "ALL")
    rows += row(unc,  "Needs Review",  "⚡", "#ff5555",  "UNCERTAIN")
    rows += "<div style='border-top:1px solid #21262d;margin:8px 0;'></div>"
    for c in range(NUM_LABELS):
        n    = CLASS_NAMES[c]
        cnt  = counts.get(n, 0)
        pct  = cnt / total * 100 if total else 0
        bar  = f"""<div style="width:{pct:.0f}%;height:3px;background:{CLASS_COLORS[n]};
                   border-radius:2px;min-width:{'2px' if cnt else '0'};"></div>"""
        rows += f"""<div style="padding:8px 10px;border-radius:7px;cursor:pointer;
            border:1px solid #21262d;margin-bottom:4px;
            {'border-color:'+CLASS_COLORS[n]+';' if filt==n else ''}">
            <div style="display:flex;align-items:center;gap:8px;margin-bottom:3px;">
                <span style="font-size:13px;">{CLASS_ICONS[n]}</span>
                <span style="font-family:'DM Sans',sans-serif;font-size:11px;
                    color:#c9d1d9;flex:1;white-space:nowrap;overflow:hidden;
                    text-overflow:ellipsis;">{n.replace('_',' ')}</span>
                <span style="font-family:'Space Mono',monospace;font-size:11px;
                    color:{CLASS_COLORS[n]};font-weight:700;">{cnt}</span>
            </div>
            <div style="background:#21262d;border-radius:2px;height:3px;">{bar}</div>
        </div>"""

    return f"""{FONTS}<div style="background:#0d1117;padding:4px;">
        <div style="font-family:'Space Mono',monospace;font-size:9px;color:#3a4a5a;
            letter-spacing:2px;text-transform:uppercase;margin-bottom:8px;">
            Categories</div>{rows}</div>"""

def build_feed():
    vis  = classified_tweets[:app_state["visible_count"]]
    filt = app_state["active_filter"]
    sel  = app_state["selected_id"]

    if filt == "UNCERTAIN":
        shown = [t for t in vis if t["uncertain"]]
    elif filt == "ALL":
        shown = vis
    else:
        shown = [t for t in vis if t["pred_name"] == filt]

    if not shown:
        msg = "AWAITING INCOMING TWEETS" if not vis else "NO TWEETS IN THIS CATEGORY"
        return f"""{FONTS}<div style="background:#0d1117;height:480px;display:flex;
            align-items:center;justify-content:center;">
            <div style="text-align:center;color:#3a4a5a;">
                <div style="font-size:28px;margin-bottom:10px;">📡</div>
                <div style="font-family:'Space Mono',monospace;font-size:11px;
                    letter-spacing:2px;">{msg}</div>
            </div></div>"""

    cards = ""
    for t in reversed(shown):
        color  = CLASS_COLORS[t["pred_name"]]
        cc     = conf_color(t["confidence"])
        icon   = CLASS_ICONS[t["pred_name"]]
        is_sel = t["id"] == sel
        sel_st = f"border-color:{color};background:#0d1520;" if is_sel else ""
        unc_t  = """<span style="background:rgba(255,85,85,0.15);border:1px solid
            rgba(255,85,85,0.4);border-radius:3px;padding:1px 5px;font-size:9px;
            color:#ff5555;font-family:'Space Mono',monospace;margin-left:5px;">
            REVIEW</span>""" if t["uncertain"] else ""
        prev   = t["text"][:110] + "…" if len(t["text"]) > 110 else t["text"]

        cards += f"""<div data-id="{t['id']}" class="tweet-card"
            style="background:#161b22;border:1px solid #21262d;border-radius:9px;
            padding:11px 13px;margin-bottom:7px;cursor:pointer;{sel_st}
            position:relative;overflow:hidden;">
            <div style="position:absolute;left:0;top:0;bottom:0;width:3px;
                background:{color};border-radius:9px 0 0 9px;"></div>
            <div style="display:flex;align-items:center;gap:7px;
                margin-bottom:5px;padding-left:7px;">
                <span style="font-size:12px;">{icon}</span>
                <span style="font-family:'DM Sans',sans-serif;font-size:11px;
                    font-weight:600;color:{color};">
                    {t['pred_name'].replace('_',' ')}</span>
                {unc_t}
                <span style="margin-left:auto;font-family:'Space Mono',monospace;
                    font-size:11px;color:{cc};font-weight:700;">
                    {t['confidence']:.0%}</span>
            </div>
            <div style="font-family:'DM Sans',sans-serif;font-size:12px;
                color:#8b949e;line-height:1.5;padding-left:7px;">{prev}</div>
        </div>"""

    return f"""{FONTS}
    <style>
    .tweet-card:hover {{ border-color: #30363d !important;
                         background: #1a2030 !important; }}
    </style>
    <div style="background:#0d1117;padding:6px;height:480px;overflow-y:auto;">
        {cards}
    </div>"""

def build_detail(tweet_id):
    if tweet_id is None:
        return f"""{FONTS}<div style="background:#0d1117;height:480px;display:flex;
            align-items:center;justify-content:center;border:1px solid #21262d;
            border-radius:12px;">
            <div style="text-align:center;color:#3a4a5a;">
                <div style="font-size:24px;margin-bottom:8px;">👆</div>
                <div style="font-family:'Space Mono',monospace;font-size:10px;
                    letter-spacing:2px;">SELECT A TWEET TO ANALYSE</div>
            </div></div>"""

    tweet = next((t for t in classified_tweets if t["id"] == tweet_id), None)
    if not tweet:
        return build_detail(None)

    # Compute attributions on demand
    if "tokens" not in tweet:
        toks, attrs = get_attributions(tweet["text"], tweet["pred_label"])
        tweet["tokens"]       = toks
        tweet["attributions"] = attrs.tolist()

    color = CLASS_COLORS[tweet["pred_name"]]
    icon  = CLASS_ICONS[tweet["pred_name"]]
    label = tweet["pred_name"].replace("_", " ").title()
    conf  = tweet["confidence"]
    cc    = conf_color(conf)

    # Badge
    if tweet["uncertain"]:
        badge = f"""<div style="background:rgba(255,85,85,0.1);border:1px solid
            rgba(255,85,85,0.4);border-radius:7px;padding:9px 12px;margin:10px 0;
            font-family:'Space Mono',monospace;font-size:11px;font-weight:700;
            color:#ff5555;display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
            ⚡ LOW CONFIDENCE — routed to human review
            <span style="font-weight:400;font-size:10px;opacity:0.8;">
            {conf:.1%} &lt; threshold {OPTIMAL_THRESHOLD:.1%}</span></div>"""
    else:
        badge = f"""<div style="background:rgba(0,229,160,0.07);border:1px solid
            rgba(0,229,160,0.3);border-radius:7px;padding:9px 12px;margin:10px 0;
            font-family:'Space Mono',monospace;font-size:11px;font-weight:700;
            color:#00e5a0;display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
            ✅ HIGH CONFIDENCE — automated classification reliable
            <span style="font-weight:400;font-size:10px;opacity:0.8;">
            {conf:.1%} &gt; threshold {OPTIMAL_THRESHOLD:.1%}</span></div>"""

    # Token highlights
    skip  = {tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token}
    t_html = ""
    for tok, score in zip(tweet["tokens"], tweet["attributions"]):
        if tok in skip: continue
        dtok = tok.replace("Ġ", " ").replace("Ċ", " ")
        a    = min(0.15 + abs(score) * 0.75, 0.92)
        bg   = f"rgba(0,210,120,{a:.2f})" if score > 0 else f"rgba(255,80,80,{a:.2f})"
        t_html += (f'<span style="background:{bg};padding:3px 2px;margin:1px 0;'
                   f'border-radius:3px;font-size:13px;line-height:2.4;'
                   f'font-family:DM Sans,sans-serif;cursor:default;" '
                   f'title="score:{score:.3f}">{dtok}</span>')

    # Top-5 prob bars
    top5  = np.argsort(tweet["probs"])[::-1][:5]
    pbars = ""
    for c in top5:
        pct   = tweet["probs"][c] * 100
        top   = c == tweet["pred_label"]
        bc    = CLASS_COLORS[CLASS_NAMES[c]] if top else "#2a3a4a"
        tc    = CLASS_COLORS[CLASS_NAMES[c]] if top else "#4a5a6a"
        pbars += f"""<div style="display:grid;grid-template-columns:190px 1fr 44px;
            align-items:center;gap:8px;margin-bottom:6px;">
            <div style="font-size:11px;color:{tc};font-weight:{'600' if top else '400'};
                font-family:'DM Sans',sans-serif;white-space:nowrap;overflow:hidden;
                text-overflow:ellipsis;">{CLASS_ICONS[CLASS_NAMES[c]]} {CLASS_NAMES[c].replace('_',' ')}</div>
            <div style="background:#1c2330;border-radius:3px;height:6px;overflow:hidden;">
                <div style="width:{pct:.1f}%;height:100%;background:{bc};
                    border-radius:3px;min-width:2px;"></div></div>
            <div style="font-family:'Space Mono',monospace;font-size:10px;
                color:{tc};text-align:right;">{pct:.1f}%</div>
        </div>"""

    true_l  = CLASS_NAMES[tweet["true_label"]]
    correct = tweet["pred_name"] == true_l
    ms      = "color:#00e5a0;" if correct else "color:#ff5555;"
    mt      = "✓ Correct" if correct else "✗ Incorrect"

    return f"""{FONTS}
<div style="background:#0d1117;border:1px solid #21262d;border-radius:12px;
    padding:16px;height:480px;overflow-y:auto;">
    <div style="display:flex;align-items:center;gap:10px;margin-bottom:4px;">
        <span style="font-size:26px;">{icon}</span>
        <div style="flex:1;">
            <div style="font-family:'Space Mono',monospace;font-size:15px;
                font-weight:700;color:{color};">{label}</div>
            <div style="font-size:10px;color:#4a5a6a;margin-top:2px;
                font-family:'Space Mono',monospace;">
                Ground truth: {true_l.replace('_',' ')}
                &nbsp;<span style="{ms}font-weight:700;">{mt}</span></div>
        </div>
        <div style="text-align:center;">
            <div style="font-family:'Space Mono',monospace;font-size:22px;
                font-weight:700;color:{cc};">{conf:.0%}</div>
            <div style="font-size:9px;color:#4a5a6a;font-family:'Space Mono',
                monospace;letter-spacing:1px;text-transform:uppercase;">confidence</div>
        </div>
    </div>
    {badge}
    <div style="font-size:9px;font-family:'Space Mono',monospace;letter-spacing:2px;
        color:#3a4a5a;text-transform:uppercase;margin-bottom:5px;">Tweet</div>
    <div style="background:#161b22;border:1px solid #21262d;border-radius:7px;
        padding:10px 12px;font-size:13px;line-height:1.6;color:#c9d1d9;
        margin-bottom:12px;font-family:'DM Sans',sans-serif;">{tweet['text']}</div>
    <div style="font-size:9px;font-family:'Space Mono',monospace;letter-spacing:2px;
        color:#3a4a5a;text-transform:uppercase;margin-bottom:5px;">
        Token Attributions</div>
    <div style="background:#161b22;border:1px solid #21262d;border-radius:7px;
        padding:10px 12px;margin-bottom:12px;line-height:2.4;word-wrap:break-word;">
        {t_html}
        <div style="display:flex;gap:14px;margin-top:8px;padding-top:7px;
            border-top:1px solid #21262d;">
            <div style="display:flex;align-items:center;gap:5px;font-size:9px;
                color:#3a4a5a;font-family:'Space Mono',monospace;">
                <div style="width:9px;height:9px;border-radius:2px;
                    background:rgba(0,210,120,0.7);"></div>supports</div>
            <div style="display:flex;align-items:center;gap:5px;font-size:9px;
                color:#3a4a5a;font-family:'Space Mono',monospace;">
                <div style="width:9px;height:9px;border-radius:2px;
                    background:rgba(255,80,80,0.7);"></div>opposes</div>
        </div>
    </div>
    <div style="font-size:9px;font-family:'Space Mono',monospace;letter-spacing:2px;
        color:#3a4a5a;text-transform:uppercase;margin-bottom:5px;">
        Top-5 Probabilities</div>
    <div style="background:#161b22;border:1px solid #21262d;border-radius:7px;
        padding:10px 12px;">{pbars}</div>
</div>"""

In [8]:
# ── Cell 9: Gradio 5 App ──────────────────────────────────────────────────────
CSS = """
.gradio-container { background: #0d1117 !important; }
button.primary { background: linear-gradient(135deg,#00e5a0,#0070f3) !important;
    border: none !important; color: #0d1117 !important;
    font-family: 'Space Mono',monospace !important; font-weight: 700 !important; }
button.secondary { background: #161b22 !important;
    border: 1px solid #30363d !important; color: #8b949e !important;
    font-family: 'Space Mono',monospace !important; }
footer { display: none !important; }
"""

with gr.Blocks(css=CSS, title="Crisis Dashboard") as demo:

    # Header
    gr.HTML(f"""{FONTS}
    <div style="background:#0d1117;padding:16px 20px 14px;
        border-bottom:1px solid #21262d;margin-bottom:14px;">
        <div style="display:flex;align-items:center;gap:12px;flex-wrap:wrap;">
            <span style="font-size:20px;">⚡</span>
            <div>
                <div style="font-family:'Space Mono',monospace;font-size:15px;
                    font-weight:700;color:#e6edf3;">CRISIS INFORMATION DASHBOARD</div>
                <div style="font-size:11px;color:#6b7a8a;font-family:'DM Sans',sans-serif;
                    margin-top:2px;">Real-time Disaster Tweet Classification &nbsp;·&nbsp;
                    Active Event Monitor</div>
            </div>
            <div style="margin-left:auto;display:flex;gap:8px;flex-wrap:wrap;">
                <div style="background:#161b22;border:1px solid #21262d;
                    border-radius:20px;padding:3px 10px;font-size:10px;
                    font-family:'Space Mono',monospace;color:#00e5a0;">
                    RoBERTa · DeBERTa · ELECTRA Ensemble</div>
                <div style="background:rgba(0,229,160,0.1);border:1px solid
                    rgba(0,229,160,0.3);border-radius:20px;padding:3px 10px;
                    font-size:10px;font-family:'Space Mono',monospace;color:#00e5a0;">
                    ● LIVE</div>
            </div>
        </div>
    </div>""")

    # Metrics
    metrics_out = gr.HTML(build_metrics())

    # Controls
    with gr.Row():
        start_btn = gr.Button("▶  START FEED",  variant="primary",   scale=1)
        pause_btn = gr.Button("⏸  PAUSE",        variant="secondary", scale=1)
        reset_btn = gr.Button("↺  RESET",         variant="secondary", scale=1)
        speed_sl  = gr.Slider(0.5, 4.0, value=1.5, step=0.5,
                              label="Speed (seconds between tweets)", scale=3)

    # Filter buttons row
    with gr.Row():
        filter_btns = []
        all_btn = gr.Button("🌐 ALL",          variant="secondary", scale=1)
        unc_btn = gr.Button("⚡ Needs Review", variant="secondary", scale=1)
        filter_btns = [all_btn, unc_btn]
        for c in range(NUM_LABELS):
            n = CLASS_NAMES[c]
            filter_btns.append(
                gr.Button(f"{CLASS_ICONS[n]} {n.replace('_',' ')[:12]}",
                          variant="secondary", scale=1)
            )

    # Main 3-column layout
    with gr.Row():
        with gr.Column(scale=1, min_width=180):
            sidebar_out = gr.HTML(build_sidebar())
        with gr.Column(scale=2):
            feed_out = gr.HTML(build_feed())
        with gr.Column(scale=2):
            detail_out = gr.HTML(build_detail(None))

    # Tweet selector (hidden number input — user clicks feed cards)
    tweet_selector = gr.Number(value=-1, visible=False, label="selected_tweet_id")

    # Footer
    gr.HTML(f"""{FONTS}
    <div style="text-align:center;padding:14px 0 6px;border-top:1px solid #21262d;
        margin-top:10px;font-family:'Space Mono',monospace;font-size:9px;
        color:#3a4a5a;letter-spacing:1px;">
        B.Tech Final Year Project &nbsp;·&nbsp;
        Models: RoBERTa · DeBERTa · ELECTRA &nbsp;·&nbsp;
        Dataset: HumAID (QCRI) &nbsp;·&nbsp;
        Ensemble Macro F1: 0.769 &nbsp;·&nbsp;
        Confidence Threshold: {OPTIMAL_THRESHOLD:.2f}
    </div>""")

    # ── Timer — advances feed automatically ───────────────────────────────────
    timer = gr.Timer(value=1.5, active=False)

    def tick(speed):
        if not app_state["running"]:
            return build_sidebar(), build_feed(), build_metrics()
        if app_state["visible_count"] < len(classified_tweets):
            app_state["visible_count"] += 1
        return build_sidebar(), build_feed(), build_metrics()

    timer.tick(fn=tick, inputs=[speed_sl],
               outputs=[sidebar_out, feed_out, metrics_out])

    # ── Button handlers ───────────────────────────────────────────────────────
    def start():
        app_state["running"] = True
        return gr.Timer(active=True), build_sidebar(), build_feed(), build_metrics()

    def pause():
        app_state["running"] = False
        return gr.Timer(active=False), build_sidebar(), build_feed(), build_metrics()

    def reset():
        app_state["running"]       = False
        app_state["visible_count"] = 0
        app_state["selected_id"]   = None
        app_state["active_filter"] = "ALL"
        return (gr.Timer(active=False), build_sidebar(),
                build_feed(), build_detail(None), build_metrics())

    start_btn.click(fn=start,
                    outputs=[timer, sidebar_out, feed_out, metrics_out])
    pause_btn.click(fn=pause,
                    outputs=[timer, sidebar_out, feed_out, metrics_out])
    reset_btn.click(fn=reset,
                    outputs=[timer, sidebar_out, feed_out, detail_out, metrics_out])

    # ── Filter button handlers ────────────────────────────────────────────────
    def make_filter_fn(filter_val):
        def fn():
            app_state["active_filter"] = filter_val
            return build_sidebar(), build_feed()
        return fn

    all_btn.click(fn=make_filter_fn("ALL"),
                  outputs=[sidebar_out, feed_out])
    unc_btn.click(fn=make_filter_fn("UNCERTAIN"),
                  outputs=[sidebar_out, feed_out])
    for i, c in enumerate(range(NUM_LABELS)):
        filter_btns[i + 2].click(
            fn=make_filter_fn(CLASS_NAMES[c]),
            outputs=[sidebar_out, feed_out],
        )

    # ── Tweet selection via dropdown ──────────────────────────────────────────
    # Since Gradio 5 doesn't allow clicking arbitrary HTML elements to trigger
    # Python, we use a Dropdown of visible tweet IDs as selector
    gr.HTML(f"""{FONTS}
    <div style="font-family:'Space Mono',monospace;font-size:9px;color:#3a4a5a;
        letter-spacing:2px;text-transform:uppercase;margin-top:10px;">
        Select tweet to analyse ↓</div>""")

    tweet_dropdown = gr.Dropdown(
        choices=[],
        label="Select Tweet ID to analyse in detail panel",
        interactive=True,
    )

    def update_dropdown():
        vis     = classified_tweets[:app_state["visible_count"]]
        choices = [f"[{t['id']}] {t['text'][:80]}…" if len(t['text']) > 80
                   else f"[{t['id']}] {t['text']}"
                   for t in reversed(vis)]
        return gr.Dropdown(choices=choices)

    def on_select(choice):
        if not choice:
            return build_detail(None)
        tweet_id = int(choice.split("]")[0].replace("[", "").strip())
        app_state["selected_id"] = tweet_id
        return build_detail(tweet_id)

    # Refresh dropdown when feed updates
    timer.tick(fn=update_dropdown, outputs=[tweet_dropdown])
    start_btn.click(fn=update_dropdown, outputs=[tweet_dropdown])

    tweet_dropdown.change(fn=on_select, inputs=[tweet_dropdown],
                          outputs=[detail_out])


/tmp/ipykernel_1358/4288103070.py:13: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="Crisis Dashboard") as demo:


In [9]:
# ── Cell 10: Launch ───────────────────────────────────────────────────────────
print("\nLaunching Crisis Dashboard (Gradio 5)...")
demo.launch(share=True, debug=False)


Launching Crisis Dashboard (Gradio 5)...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7422d885a10ec2540.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
